In [5]:
import sympy
from sympy import symbols, Matrix, I, exp, S, init_printing, diag
from sympy.physics.quantum import TensorProduct
from sympy import latex
from sympy import PermutationMatrix
from sympy.combinatorics import Permutation

init_printing(use_latex='mathjax')

theta = symbols('theta')
theta_1, theta_2, theta_3 = symbols('theta_1 theta_2 theta_3')

In [6]:
import re
def show_mitex(s):
    print(f"#mitex(`{latex(s)}`)")

In [7]:
# 使用 S(1) 解决 Literal[1] 与 Pow 之间的除法类型冲突
H = (S(1) / sympy.sqrt(2)) * Matrix([[1, 1], [1, -1]])
I2 = Matrix([[1, 0], [0, 1]])

# 定义 CP(theta) 矩阵
# 该门仅在 |11> 态引入相位 exp(i*theta)
CP = Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, exp(I * theta)]
])

# 4. 构造 H1 和 H2
# H1 = H ⊗ I (作用在第一个 qubit)
# H2 = I ⊗ H (作用在第二个 qubit)
H1 = TensorProduct(H, I2)
H2 = TensorProduct(I2, H)

# 5. 计算复合矩阵 M = H2 * CP * H1
# 注意：在量子力学算子表示中，顺序通常是左乘。
# 如果你的物理含义是“先H1，再CP，后H2”，则顺序为 H2 * CP * H1
M = H2 * CP * H1

# 6. 化简结果
M_simplified = sympy.simplify(M)

# 7. 在 ipynb 中打印（会自动触发 LaTeX 渲染）
M_simplified

⎡1/2   1/2    1/2    1/2  ⎤
⎢                         ⎥
⎢1/2   -1/2   1/2    -1/2 ⎥
⎢                         ⎥
⎢       ⅈ⋅θ           ⅈ⋅θ ⎥
⎢      ℯ            -ℯ    ⎥
⎢1/2   ────   -1/2  ──────⎥
⎢       2             2   ⎥
⎢                         ⎥
⎢       ⅈ⋅θ           ⅈ⋅θ ⎥
⎢     -ℯ             ℯ    ⎥
⎢1/2  ──────  -1/2   ──── ⎥
⎣       2             2   ⎦

In [22]:
# 3. 定义基础算门
H = (S(1) / sympy.sqrt(2)) * Matrix([[1, 1], [1, -1]])
I2 = Matrix([[1, 0], [0, 1]])

# 4. 构造 H1, H2, H3 (作用于 3 个 qubit，按照 q0 ⊗ q1 ⊗ q2 顺序)
H1 = TensorProduct(H, I2, I2)
H2 = TensorProduct(I2, H, I2)
H3 = TensorProduct(I2, I2, H)

# 5. 构造 8x8 的 CP 门（对角矩阵）
# 状态索引对应：
# 0: 000, 1: 001, 2: 010, 3: 011
# 4: 100, 5: 101, 6: 110, 7: 111

# CP_01: 仅在 |110> 和 |111> 加上 exp(I*theta_1)
CP_01 = diag(1, 1, 1, 1, 1, 1, exp(I * theta_1), exp(I * theta_1))

# CP_12: 仅在 |011> 和 |111> 加上 exp(I*theta_2)
CP_12 = diag(1, 1, 1, exp(I * theta_2), 1, 1, 1, exp(I * theta_2))

# CP_02: 仅在 |101> 和 |111> 加上 exp(I*theta_3)
CP_02 = diag(1, 1, 1, 1, 1, exp(I * theta_3), 1, exp(I * theta_3))

# 6. 计算复合矩阵 M
# 运算顺序：H3 * CP(theta_3, [0,2]) * CP(theta_2, [1,2]) * H2 * CP(theta_1, [0,1]) * H1
M = H3 * CP_02 * CP_12 * H2 * CP_01 * H1

bit_reversal_perm = Permutation([0, 4, 2, 6, 1, 5, 3, 7])
P = PermutationMatrix(bit_reversal_perm).as_explicit()

# 7. 化简结果
M_simplified = sympy.simplify(M*4/sympy.sqrt(2))

# 8. 在 ipynb 中打印
# show_mitex(M_simplified)
display(M_simplified) 
display(sympy.simplify((P@M_simplified).doit()))
M_simplified = sympy.simplify((P@M_simplified).doit())
show_mitex(M_simplified)

⎡1        1          1             1           1         1          1          ↪
⎢                                                                              ↪
⎢1       -1          1             -1          1        -1          1          ↪
⎢                                                                              ↪
⎢        ⅈ⋅θ₂                      ⅈ⋅θ₂                 ⅈ⋅θ₂                   ↪
⎢1      ℯ            -1          -ℯ            1       ℯ            -1         ↪
⎢                                                                              ↪
⎢        ⅈ⋅θ₂                     ⅈ⋅θ₂                  ⅈ⋅θ₂                   ↪
⎢1     -ℯ            -1          ℯ             1      -ℯ            -1         ↪
⎢                                                                              ↪
⎢        ⅈ⋅θ₃       ⅈ⋅θ₁       ⅈ⋅(θ₁ + θ₃)              ⅈ⋅θ₃        ⅈ⋅θ₁       ↪
⎢1      ℯ          ℯ          ℯ                -1     -ℯ          -ℯ        -ℯ ↪
⎢                           

⎡1        1          1             1           1         1          1          ↪
⎢                                                                              ↪
⎢        ⅈ⋅θ₃       ⅈ⋅θ₁       ⅈ⋅(θ₁ + θ₃)              ⅈ⋅θ₃        ⅈ⋅θ₁       ↪
⎢1      ℯ          ℯ          ℯ                -1     -ℯ          -ℯ        -ℯ ↪
⎢                                                                              ↪
⎢        ⅈ⋅θ₂                      ⅈ⋅θ₂                 ⅈ⋅θ₂                   ↪
⎢1      ℯ            -1          -ℯ            1       ℯ            -1         ↪
⎢                                                                              ↪
⎢    ⅈ⋅(θ₂ + θ₃)     ⅈ⋅θ₁    ⅈ⋅(θ₁ + θ₂ + θ₃)        ⅈ⋅(θ₂ + θ₃)   ⅈ⋅θ₁    ⅈ⋅( ↪
⎢1  ℯ              -ℯ      -ℯ                  -1  -ℯ             ℯ       ℯ    ↪
⎢                                                                              ↪
⎢1       -1          1             -1          1        -1          1          ↪
⎢                           

#mitex(`\left[\begin{matrix}1 & 1 & 1 & 1 & 1 & 1 & 1 & 1\\1 & e^{i \theta_{3}} & e^{i \theta_{1}} & e^{i \left(\theta_{1} + \theta_{3}\right)} & -1 & - e^{i \theta_{3}} & - e^{i \theta_{1}} & - e^{i \left(\theta_{1} + \theta_{3}\right)}\\1 & e^{i \theta_{2}} & -1 & - e^{i \theta_{2}} & 1 & e^{i \theta_{2}} & -1 & - e^{i \theta_{2}}\\1 & e^{i \left(\theta_{2} + \theta_{3}\right)} & - e^{i \theta_{1}} & - e^{i \left(\theta_{1} + \theta_{2} + \theta_{3}\right)} & -1 & - e^{i \left(\theta_{2} + \theta_{3}\right)} & e^{i \theta_{1}} & e^{i \left(\theta_{1} + \theta_{2} + \theta_{3}\right)}\\1 & -1 & 1 & -1 & 1 & -1 & 1 & -1\\1 & - e^{i \theta_{3}} & e^{i \theta_{1}} & - e^{i \left(\theta_{1} + \theta_{3}\right)} & -1 & e^{i \theta_{3}} & - e^{i \theta_{1}} & e^{i \left(\theta_{1} + \theta_{3}\right)}\\1 & - e^{i \theta_{2}} & -1 & e^{i \theta_{2}} & 1 & - e^{i \theta_{2}} & -1 & e^{i \theta_{2}}\\1 & - e^{i \left(\theta_{2} + \theta_{3}\right)} & - e^{i \theta_{1}} & e^{i \left(\theta_{1} 

In [20]:
expr = (M_simplified[0,2] + M_simplified[4,2])**2 - (M_simplified[2,2] + M_simplified[6,2])**2

res = sympy.simplify(sympy.expand(expr) *8)
display(res)
show_mitex(res)

0

#mitex(`0`)


In [19]:
expr = (M_simplified[0,1] + M_simplified[4,1])**2 - (M_simplified[3,1] + M_simplified[7,1])**2
res = sympy.simplify(sympy.expand(expr)*8)
display(res)
show_mitex(res)

0

#mitex(`0`)


In [11]:
expr = (M_simplified[0,3] + M_simplified[4,3])**2 - (M_simplified[3,2] + M_simplified[7,2])**2
res = sympy.simplify(sympy.expand(expr)*8)
display(res)
show_mitex(res)

     2⋅ⅈ⋅θ₁       ⅈ⋅θ₁      2⋅ⅈ⋅(θ₁ + θ₃)       ⅈ⋅(θ₁ + θ₃)
- 8⋅ℯ       - 16⋅ℯ     + 8⋅ℯ              + 16⋅ℯ           

#mitex(`- 8 e^{2 i \theta_{1}} - 16 e^{i \theta_{1}} + 8 e^{2 i \left(\theta_{1} + \theta_{3}\right)} + 16 e^{i \left(\theta_{1} + \theta_{3}\right)}`)


In [12]:
M_simplified[0,1] + M_simplified[3,1]

     ⅈ⋅θ₂
1 - ℯ    

In [13]:
import qiskit
import numpy as np
qc = qiskit.QuantumCircuit(3)
qc.h(0)
qc.cp(np.pi, 0,1)